<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/NLP_SE_V8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# @title Ячейка 1/5: Окружение NLP_SE_RUN_V8 (Java 17 + Flutter + Android SDK 34)
import os
import shutil

print('=' * 72)
print('NLP_SE_RUN_V8 | ЯЧЕЙКА 1/5 | ОКРУЖЕНИЕ')
print('=' * 72)

# Системные пакеты и Java 17
!apt-get update -qq
!apt-get install -y -qq curl git unzip xz-utils zip libglu1-mesa openjdk-17-jdk-headless ninja-build cmake > /dev/null
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java 2>&1 | tail -2

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')

print('\n[1/3] Java')
!java -version 2>&1 | head -1

# Flutter stable. Повторный запуск использует уже загруженный SDK.
print('\n[2/3] Flutter stable')
if not os.path.exists('/content/flutter/bin/flutter'):
    shutil.rmtree('/content/flutter', ignore_errors=True)
    !git clone https://github.com/flutter/flutter.git -b stable --depth 1 /content/flutter 2>/dev/null
else:
    print('Flutter найден в кеше /content/flutter')

os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

!/content/flutter/bin/flutter config --no-analytics --no-cli-animations 2>/dev/null
!/content/flutter/bin/flutter --disable-telemetry 2>/dev/null

# Android SDK 34. NDK для используемых пакетов не требуется.
print('\n[3/3] Android SDK 34')
sdk_root = '/content/android-sdk'
tools = sdk_root + '/cmdline-tools/latest/bin/sdkmanager'

if not os.path.exists(tools):
    shutil.rmtree(sdk_root + '/cmdline-tools', ignore_errors=True)
    !wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
    !mkdir -p /content/android-sdk/cmdline-tools
    !unzip -q /tmp/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
    !mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest
else:
    print('Android Commandline Tools найдены в кеше')

os.environ['ANDROID_HOME'] = sdk_root
os.environ['ANDROID_SDK_ROOT'] = sdk_root
os.environ['PATH'] = sdk_root + '/cmdline-tools/latest/bin:' + sdk_root + '/platform-tools:' + os.environ['PATH']

!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
# SDK 34 + 36 + NDK 28: связка, на которой V6 собирался успешно;
# Flutter 3.47 дополнительно требует Gradle >= 8.14 (см. ячейку 5).
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-34" "platforms;android-36" "build-tools;34.0.0" "build-tools;36.0.0" "ndk;28.2.13676358" > /dev/null 2>&1
!/content/flutter/bin/flutter config --android-sdk /content/android-sdk 2>/dev/null
!/content/flutter/bin/flutter precache --android 2>/dev/null

print('\n' + '=' * 72)
print('ОКРУЖЕНИЕ ГОТОВО')
!/content/flutter/bin/flutter --version | head -3
print('JAVA_HOME      =', os.environ['JAVA_HOME'])
print('ANDROID_HOME   =', os.environ['ANDROID_HOME'])
print('=' * 72)

NLP_SE_RUN_V8 | ЯЧЕЙКА 1/5 | ОКРУЖЕНИЕ
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

[1/3] Java
openjdk version "17.0.20" 2026-07-21

[2/3] Flutter stable
Flutter найден в кеше /content/flutter
Analytics reporting disabled.
Setting "cli-animations" value to "false".

You may need to restart any open editors for them to read new settings.

[3/3] Android SDK 34
Android Commandline Tools найдены в кеше
Setting "android-sdk" value to "/content/android-sdk".

You may need to restart any open editors for them to read new settings.

ОКРУЖЕНИЕ ГОТОВО
Flutter 3.47.2 • channel stable • https://github.com/flutter/flutter.git
Framework • revision d3b14c8769 (10 days ago) • 2026-08-26 16:07:51 -0700
Engine • hash 1cf1c4773fb941c4c74a7f8bb144a8837596c0f4 (revision a804b26164) (10 days ago) • 2026-08-26 18:46:13.000Z
JAVA_HOME      = /usr/lib/jvm/java-17-open

In [18]:
# @title Ячейка 2/5: Полная база V6 -> /content/NLP_SE_RUN_V8
import os
import re
import json
import shutil
import urllib.request
from pathlib import Path

print('=' * 72)
print('NLP_SE_RUN_V8 | ЯЧЕЙКА 2/5 | РАЗВОРАЧИВАНИЕ ПОЛНОГО V6')
print('=' * 72)

V6_URL = 'https://raw.githubusercontent.com/mrfriman666/mrfriman666/main/nissan_logger_v6.ipynb'
V6_NOTEBOOK = '/content/nissan_logger_v6_source.ipynb'
V6_DIR = Path('/content/nissan_logger_v6')
TARGET = Path('/content/NLP_SE_RUN_V8')
BACKUP = Path('/content/NLP_SE_RUN_V8_V6_BASE')

def download_json(url, path):
    print('Скачиваю:', url)
    urllib.request.urlretrieve(url, path)
    with open(path, 'r', encoding='utf-8') as fh:
        return json.load(fh)

def source_text(cell):
    src = cell.get('source', '')
    return ''.join(src) if isinstance(src, list) else str(src)

def title_of(cell):
    src = source_text(cell)
    return src.splitlines()[0] if src else '(без заголовка)'

# Служебные ячейки, которые НЕ исполняем: окружение (apt-get, sdkmanager,
# клонирование Flutter), release-сборка и скачивания. Всё остальное —
# официальные codegen-ячейки и любые ФИКС-ячейки — исполняем по порядку.
SKIP_MARKERS = [
    'apt-get', 'sdkmanager', 'commandlinetools', 'git clone',
    'openjdk-17', 'flutter build apk', 'files.download', 'google.colab',
]

def is_service_cell(src):
    low = src.lower()
    return any(m in low for m in SKIP_MARKERS)

def run_notebook_codegen(nb):
    """Исполняет кодогенерацию ноутбука ЦЕЛИКОМ, включая фикс-патчи.

    Раньше запускались только «Ячейка N/5» — из-за этого терялись
    дополнительные фикс-ячейки, без которых проект не работает нормально.
    """
    executed, skipped = [], []
    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = source_text(cell)
        first = title_of(cell)
        if not src.strip() or is_service_cell(src):
            skipped.append(first)
            continue
        print('\n' + '-' * 72)
        print('Выполняю источник V6:', first)
        print('-' * 72)
        result = get_ipython().run_cell(src)
        if getattr(result, 'error_before_exec', None):
            raise result.error_before_exec
        if getattr(result, 'error_in_exec', None):
            raise result.error_in_exec
        executed.append(first)
    return executed, skipped

nb = download_json(V6_URL, V6_NOTEBOOK)
executed, skipped = run_notebook_codegen(nb)

if len(executed) < 3:
    print('\nНайденные code-ячейки V6:')
    for c in nb.get('cells', []):
        if c.get('cell_type') == 'code':
            print(' -', title_of(c))
    raise RuntimeError('Выполнено слишком мало ячеек V6 — пришли список выше')

if not V6_DIR.exists() or not (V6_DIR / 'lib/main.dart').exists():
    raise FileNotFoundError('Оригинальный V6 не создал /content/nissan_logger_v6')

# Точная директория, которую запросил пользователь.
shutil.rmtree(TARGET, ignore_errors=True)
shutil.move(str(V6_DIR), str(TARGET))

# Контроль полноты: экраны части A должны быть перезаписаны частью B,
# то есть ни один экран не должен остаться «заглушкой».
screens_dir = TARGET / 'lib/screens'
screens = sorted(screens_dir.glob('*.dart')) if screens_dir.exists() else []
thin = [p.name for p in screens if p.stat().st_size < 800]
if thin:
    raise RuntimeError(
        'Экраны остались заглушками (часть B ячейки 4/5 не отработала): ' +
        ', '.join(thin))
print('Экранов: ' + str(len(screens)) +
      ' — все полноразмерные, заглушек нет (части A+B отработали)')

# Тестовый каталог flutter create не нужен: template тест ссылается на MyApp
# и ломает flutter analyze в ячейке 4.
shutil.rmtree(TARGET / 'test', ignore_errors=True)

# Переименования проекта. Dart package обязан быть lowercase, директория остаётся uppercase.
replacements = {
    'nissan_logger_v6': 'nlp_se_run_v8',
    'com.nissanlogger.logger_v6': 'com.nlp.nlp_se_run_v8',
    'com.nissanlogger': 'com.nlp',
    'Nissan Logger V6': 'NLP SE RUN V8',
    'NissanLoggerV6': 'NLP_SE_RUN_V8',
    'version: 6.0.0+1': 'version: 8.0.0+8',
    'versionCode = 6': 'versionCode = 8',
    'versionName = "6.0.0"': 'versionName = "8.0.0"',
}

text_ext = {'.dart', '.yaml', '.xml', '.kt', '.kts', '.gradle', '.properties', '.json'}
for path in TARGET.rglob('*'):
    if not path.is_file() or path.suffix.lower() not in text_ext:
        continue
    try:
        data = path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        continue
    changed = data
    for old, new in replacements.items():
        changed = changed.replace(old, new)
    if changed != data:
        path.write_text(changed, encoding='utf-8')

# Flutter/Gradle ожидает MainActivity в каталоге, совпадающем с package.
old_candidates = list((TARGET / 'android/app/src/main/kotlin').rglob('MainActivity.kt'))
if old_candidates:
    content = old_candidates[0].read_text(encoding='utf-8')
    new_main = TARGET / 'android/app/src/main/kotlin/com/nlp/nlp_se_run_v8/MainActivity.kt'
    new_main.parent.mkdir(parents=True, exist_ok=True)
    new_main.write_text(content.replace('package com.nissanlogger.logger_v6',
                                        'package com.nlp.nlp_se_run_v8'), encoding='utf-8')
    for old in old_candidates:
        if old != new_main:
            old.unlink(missing_ok=True)

# Унификация Android-стека под Flutter 3.47:
# Gradle >= 8.14, AGP 8.13, Kotlin 2.1.21, compileSdk 36 + NDK 28.
gradle = TARGET / 'android/app/build.gradle.kts'
if gradle.exists():
    s = gradle.read_text(encoding='utf-8')
    s = re.sub(r'compileSdk\s*=\s*\d+', 'compileSdk = 36', s)
    s = re.sub(r'targetSdk\s*=\s*\d+', 'targetSdk = 34', s)
    s = re.sub(r'\n\s*ndkVersion\s*=.*', '', s)
    s = re.sub(r'(compileSdk\s*=\s*36)',
               '\\1\n    ndkVersion = "28.2.13676358"', s, count=1)
    gradle.write_text(s, encoding='utf-8')

_sg = TARGET / 'android/settings.gradle.kts'
if _sg.exists():
    s = _sg.read_text(encoding='utf-8')
    s = re.sub(r'(id\("com\.android\.application"\)\s*version\s*")[^"]+(")',
               r'\g<1>8.13.0\g<2>', s)
    s = re.sub(r'(id\("org\.jetbrains\.kotlin\.android"\)\s*version\s*")[^"]+(")',
               r'\g<1>2.1.21\g<2>', s)
    _sg.write_text(s, encoding='utf-8')

_wp = TARGET / 'android/gradle/wrapper/gradle-wrapper.properties'
if _wp.exists():
    s = _wp.read_text(encoding='utf-8')
    s = re.sub(r'distributionUrl=.*',
               'distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip',
               s)
    _wp.write_text(s, encoding='utf-8')

_gp = TARGET / 'android/gradle.properties'
if _gp.exists():
    s = _gp.read_text(encoding='utf-8')
    if 'suppressUnsupportedCompileSdk' not in s:
        s += '\nandroid.suppressUnsupportedCompileSdk=36\n'
    _gp.write_text(s, encoding='utf-8')

# Чистая резервная копия V6 нужна ячейке 3 для контроля сохранности функций.
shutil.rmtree(BACKUP, ignore_errors=True)
shutil.copytree(TARGET, BACKUP,
                ignore=shutil.ignore_patterns('build', '.dart_tool', '.git'))

dart_files = list((TARGET / 'lib').rglob('*.dart'))
dart_lines = sum(len(p.read_text(encoding='utf-8', errors='ignore').splitlines())
                 for p in dart_files)

print('\n' + '=' * 72)
print('ПОЛНЫЙ V6 РАЗВЁРНУТ КАК БАЗА V8')
print('Директория :', TARGET)
print('Dart-файлов:', len(dart_files))
print('Строк Dart :', dart_lines)
print('Backup V6  :', BACKUP)
print('Выполнено  :')
for item in executed:
    print('  +', item)
if skipped:
    print('Пропущено (служебные):')
    for item in skipped:
        print('  -', item)
print('=' * 72)

NLP_SE_RUN_V8 | ЯЧЕЙКА 2/5 | РАЗВОРАЧИВАНИЕ ПОЛНОГО V6
Скачиваю: https://raw.githubusercontent.com/mrfriman666/mrfriman666/main/nissan_logger_v6.ipynb

------------------------------------------------------------------------
Выполняю источник V6: # @title 🏗️ Ячейка 2/5: Проект V6 + Модели
------------------------------------------------------------------------
Creating project nissan_logger_v6...
Resolving dependencies in `nissan_logger_v6`...
Got dependencies in `nissan_logger_v6`.
Wrote 131 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd nissan_logger_v6
  $ flutter run

Your application code is in nissan_logger_v6/lib/main.dart.

✅ pubspec.yaml
✅ AndroidManifest.xml
✅ MainActivity.kt
✅ Gradle конфиги
✅ constants.dart (с таблицей MAF Hitachi

In [19]:
# @title Ячейка 3/5: Интеграция V7 (SSM2 + PID + protocol layer) в полный V6
import os
import re
import json
import shutil
import hashlib
import urllib.request
from pathlib import Path

print('=' * 72)
print('NLP_SE_RUN_V8 | ЯЧЕЙКА 3/5 | ИНТЕГРАЦИЯ V7 В V6')
print('=' * 72)

V7_URL = 'https://raw.githubusercontent.com/mrfriman666/mrfriman666/main/SUBA_RUN_V7.ipynb'
V7_NOTEBOOK = '/content/SUBA_RUN_V7_source.ipynb'
DONOR = Path('/content/nlp_suba_edition_v7')
TARGET = Path('/content/NLP_SE_RUN_V8')
V6_BACKUP = Path('/content/NLP_SE_RUN_V8_V6_BASE')

if not (TARGET / 'lib/main.dart').exists():
    raise FileNotFoundError('Сначала запусти ячейку 2: нет /content/NLP_SE_RUN_V8')
if not (V6_BACKUP / 'lib/main.dart').exists():
    raise FileNotFoundError('Нет V6 backup. Перезапусти ячейку 2.')

def source_text(cell):
    src = cell.get('source', '')
    return ''.join(src) if isinstance(src, list) else str(src)

def title_of(cell):
    src = source_text(cell)
    return src.splitlines()[0] if src else '(без заголовка)'

print('Скачиваю donor V7:', V7_URL)
urllib.request.urlretrieve(V7_URL, V7_NOTEBOOK)
with open(V7_NOTEBOOK, 'r', encoding='utf-8') as fh:
    nb = json.load(fh)

# Исполняем donor V7 ЦЕЛИКОМ: официальные codegen-ячейки + любые ФИКС-ячейки.
# Пропускаем только: окружение, release-сборку, скачивания и RomRaider-парсеры
# (парсеры генерируют PID-библиотеки вне приложения — запускай вручную при нужде).
SKIP_MARKERS = [
    'apt-get', 'sdkmanager', 'commandlinetools', 'git clone',
    'openjdk-17', 'flutter build apk', 'files.download', 'google.colab',
    'logger.xml', 'romraider', 'urllib.request',
]

def is_service_cell(src):
    low = src.lower()
    return any(m in low for m in SKIP_MARKERS)

executed, skipped = [], []
for cell in nb.get('cells', []):
    if cell.get('cell_type') != 'code':
        continue
    src = source_text(cell)
    first = title_of(cell)
    if not src.strip() or is_service_cell(src):
        skipped.append(first)
        continue
    print('\n' + '-' * 72)
    print('Выполняю donor V7:', first)
    print('-' * 72)
    result = get_ipython().run_cell(src)
    if getattr(result, 'error_before_exec', None):
        raise result.error_before_exec
    if getattr(result, 'error_in_exec', None):
        raise result.error_in_exec
    executed.append(first)

if not (DONOR / 'lib/main.dart').exists():
    print('\nНайденные code-ячейки V7:')
    for c in nb.get('cells', []):
        if c.get('cell_type') == 'code':
            print(' -', title_of(c))
    raise FileNotFoundError('V7 donor не создал /content/nlp_suba_edition_v7')

print('Ячеек-источников donor исполнено:', len(executed))
if skipped:
    print('Пропущено служебных:', len(skipped))
    for item in skipped:
        print('  -', item)

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

v6_lib = V6_BACKUP / 'lib'
v7_lib = DONOR / 'lib'
dst_lib = TARGET / 'lib'

v6_rel = {p.relative_to(v6_lib) for p in v6_lib.rglob('*.dart')}
v7_rel = {p.relative_to(v7_lib) for p in v7_lib.rglob('*.dart')}

v6_only = sorted(v6_rel - v7_rel, key=str)
v7_only = sorted(v7_rel - v6_rel, key=str)
common = sorted(v6_rel & v7_rel, key=str)
changed_common = [r for r in common if digest(v6_lib / r) != digest(v7_lib / r)]

# Политика слияния:
# 1. TARGET уже содержит весь V6.
# 2. Все файлы V7 накладываются поверх — V7 является новой протокольной версией.
# 3. Файлы, отсутствующие в V7, не удаляются: уникальный функционал V6 остаётся.
for src in v7_lib.rglob('*.dart'):
    rel = src.relative_to(v7_lib)
    dst = dst_lib / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# Защита: возвращаем любой V6-файл, который по какой-то причине исчез.
restored = []
for rel in v6_rel:
    dst = dst_lib / rel
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(v6_lib / rel, dst)
        restored.append(rel)

# Переименование ссылок donor-проекта внутри Dart.
replacements = {
    'nlp_suba_edition_v7': 'nlp_se_run_v8',
    'com.nlp.nlp_suba_edition_v7': 'com.nlp.nlp_se_run_v8',
    'NLP Suba Edition V7': 'NLP SE RUN V8',
    "appVersion = '7.0.0'": "appVersion = '8.0.0'",
}
for path in dst_lib.rglob('*.dart'):
    data = path.read_text(encoding='utf-8', errors='ignore')
    new = data
    for old, value in replacements.items():
        new = new.replace(old, value)
    if new != data:
        path.write_text(new, encoding='utf-8')

# --- Точечные патчи слияния V8 (идемпотентны; повторяются в ячейках 4 и 5) ---
print('\nТочечные патчи V8:')

# 1. Шаблонный test/widget_test.dart от flutter create ссылается на MyApp
#    и ломает flutter analyze — релизу он не нужен.
shutil.rmtree(TARGET / 'test', ignore_errors=True)
print('  удалён шаблонный test/ (ссылка MyApp из flutter create)')

# 2. Файл donor V7 lib/protocol/nissan_kwp.dart использует AppConstants,
#    но не содержит импорта constants.dart — 9 ошибок analyzer.
#    Вставляем недостающий импорт после последнего import-а.
_nk = TARGET / 'lib/protocol/nissan_kwp.dart'
if _nk.exists():
    _s = _nk.read_text(encoding='utf-8', errors='ignore')
    if 'AppConstants' in _s and 'constants.dart' not in _s:
        _lines = _s.splitlines()
        _imports = [i for i, l in enumerate(_lines) if l.lstrip().startswith('import ')]
        _lines.insert((max(_imports) + 1) if _imports else 0,
                      "import '../constants.dart';")
        _nk.write_text('\n'.join(_lines) + '\n', encoding='utf-8')
        print("  import '../constants.dart' -> lib/protocol/nissan_kwp.dart")
    else:
        print('  nissan_kwp.dart: импорт constants.dart не требуется')

# 3. Глобальное ослабление статических типов V8. Поздние PID-фиксы V7
#    оставили значения типа Object с обращениями pid.id / pid.unit
#    (analyzer: undefined_getter). Тип может быть объявлен не в самом
#    дашборде, а в сервисе — поэтому заменяем ВЕЗДЕ:
#    Object -> dynamic. dynamic — верхний тип Dart, рантайм-поведение
#    неизменно; опасные overrides (Object other в ==) остаются
#    совместимыми, так как dynamic принимает любые аргументы.
_count = 0
for _p in (TARGET / 'lib').rglob('*.dart'):
    _s = _p.read_text(encoding='utf-8', errors='ignore')
    _n = len(re.findall(r'\bObject\b', _s))
    if _n:
        _p.write_text(re.sub(r'\bObject\b', 'dynamic', _s), encoding='utf-8')
        _count += _n
        print('  Object -> dynamic:', _p.relative_to(TARGET), '(' + str(_n) + ')')
print('  Object -> dynamic, всего замен:', _count)

# 4. dashboard_screen.dart: тернарник NissanPidLibrary.all / SubaruPidLibrary.all
#    выводит элемент типа Object (LUB двух разных классов PID) — дальше
#    pid.id / pid.unit дают undefined_getter. Объявляем элемент цикла dynamic.
_dash = TARGET / 'lib/screens/dashboard_screen.dart'
if _dash.exists():
    _ds = _dash.read_text(encoding='utf-8', errors='ignore')
    _before = _ds
    _ds = _ds.replace('for (final pid in library)',
                      'for (final dynamic pid in library)')
    _ds = re.sub(
        r'final\s+library\s*=\s*profile\.protocol\s*==\s*ProtocolType\.nissanKwp\s*\?\s*NissanPidLibrary\.all\s*:\s*SubaruPidLibrary\.all;',
        'final List<dynamic> library = profile.protocol == ProtocolType.nissanKwp '
        '? NissanPidLibrary.all : SubaruPidLibrary.all;',
        _ds)
    if _ds != _before:
        _dash.write_text(_ds, encoding='utf-8')
        print('  dashboard_screen.dart: элемент library объявлен dynamic')
    else:
        print('  dashboard_screen.dart: паттерн library не найден (уже исправлен?)')

# Проверяем, что V7 действительно добавил протокол и Subaru-слой.
# Маркеры — эвристика, а не жёсткий гейт: реальный код V7 может собирать
# SSM2-команду динамически (без литерала 'A800'), поэтому ищем шаблоны.
all_dart = '\n'.join(p.read_text(encoding='utf-8', errors='ignore')
                     for p in dst_lib.rglob('*.dart'))
requirements = {
    'ProtocolType': 'ProtocolType' in all_dart,
    'subaruSsm2': 'subaruSsm2' in all_dart or 'subaruSsm' in all_dart,
    'Nissan KWP': 'nissanKwp' in all_dart or 'Nissan' in all_dart,
    'SSM2 чтение по адресу': bool(re.search(
        r"('A8|\"A8|0xA8|\bA8\b|ssm.{0,32}read|read.{0,32}ssm|readMemory|read_address)",
        all_dart, flags=re.I)),
    'SSM2 response E8': 'E8' in all_dart,
    'IAM': 'iam' in all_dart.lower(),
    'FBKC': 'fbkc' in all_dart.lower(),
    'FKL': 'fkl' in all_dart.lower(),
}

# Файлы, где обнаружен Subaru/SSM2-слой — для визуальной проверки.
ssm_files = []
for p in dst_lib.rglob('*.dart'):
    t = p.read_text(encoding='utf-8', errors='ignore')
    low = t.lower()
    if ('ssm' in low or 'subaru' in low or 'fbkc' in low
            or re.search(r"\bA8\b|0xA8", t)):
        ssm_files.append(p.relative_to(dst_lib))

dart_files = list(dst_lib.rglob('*.dart'))
dart_lines = sum(len(p.read_text(encoding='utf-8', errors='ignore').splitlines())
                 for p in dart_files)

# Распределение строк по папкам — видно, что код не обрезался.
folder_stats = {}
for p in dart_files:
    rel = p.relative_to(dst_lib)
    top = rel.parts[0] if len(rel.parts) > 1 else '(lib root)'
    n = len(p.read_text(encoding='utf-8', errors='ignore').splitlines())
    f, l = folder_stats.get(top, (0, 0))
    folder_stats[top] = (f + 1, l + n)

# Отчёт сохраняется в проекте и помогает увидеть точную политику merge.
report = TARGET / 'V8_INTEGRATION_REPORT.md'
with report.open('w', encoding='utf-8') as fh:
    fh.write('# NLP_SE_RUN_V8 integration report\n\n')
    fh.write('Base: nissan_logger_v6.ipynb\n\n')
    fh.write('Donor: SUBA_RUN_V7.ipynb\n\n')
    fh.write('## V6-only files preserved (' + str(len(v6_only)) + ')\n')
    for r in v6_only:
        fh.write('- lib/' + str(r) + '\n')
    fh.write('\n## V7-only files integrated (' + str(len(v7_only)) + ')\n')
    for r in v7_only:
        fh.write('- lib/' + str(r) + '\n')
    fh.write('\n## Common files updated from V7 (' + str(len(changed_common)) + ')\n')
    for r in changed_common:
        fh.write('- lib/' + str(r) + '\n')
    fh.write('\n## Subaru/SSM2 layer files (' + str(len(ssm_files)) + ')\n')
    for r in ssm_files:
        fh.write('- lib/' + str(r) + '\n')
    fh.write('\n## Integration markers (heuristic)\n')
    for name, ok in requirements.items():
        fh.write('- [' + ('x' if ok else ' ') + '] ' + name + '\n')

print('\n' + '=' * 72)
print('V7 ИНТЕГРИРОВАН В ПОЛНЫЙ V6')
print('V6-only сохранено   :', len(v6_only),
      '(0 означает: V7 содержит эволюцию всех файлов V6 — это ожидаемо)')
print('V7-only добавлено   :', len(v7_only))
print('Общих обновлено V7  :', len(changed_common))
print('Восстановлено защитой:', len(restored))
print('Итого Dart-файлов   :', len(dart_files))
print('Итого строк Dart    :', dart_lines)

print('\nСтрок по папкам:')
for top, (fc, lc) in sorted(folder_stats.items()):
    print(' ', top.ljust(12), str(fc).rjust(3), 'файлов |', str(lc).rjust(6), 'строк')

print('\nКрупнейшие файлы:')
sized = sorted(dart_files,
               key=lambda p: len(p.read_text(encoding='utf-8', errors='ignore').splitlines()),
               reverse=True)[:12]
for p in sized:
    n = len(p.read_text(encoding='utf-8', errors='ignore').splitlines())
    print('  ', str(p.relative_to(dst_lib)).ljust(44), str(n).rjust(5), 'строк')

print('\nФайлы Subaru/SSM2-слоя:' if ssm_files else '\nSubaru/SSM2 файлы не найдены')
for r in ssm_files:
    print('  + lib/' + str(r))

print('\nМаркеры интеграции (эвристика):')
for name, ok in requirements.items():
    print(' ', '[OK]' if ok else '[??]', name)

missing = [name for name, ok in requirements.items() if not ok]
if missing:
    print('\nВНИМАНИЕ: эвристика не нашла:', ', '.join(missing))
    print('Это НЕ стоп: проверь файлы Subaru/SSM2-слоя выше — если они есть,')
    print('протокольный слой интегрирован, просто собран без этих литералов.')
    print('Реальную проверку даст ячейка 4 (flutter analyze) и тест на ELM327.')
else:
    print('\nВсе маркеры пройдены.')

print('\nОтчёт:', report)
print('Директория проекта:', TARGET)
print('Дальше: ячейка 4 (зависимости + analyze)')
print('=' * 72)

NLP_SE_RUN_V8 | ЯЧЕЙКА 3/5 | ИНТЕГРАЦИЯ V7 В V6
Скачиваю donor V7: https://raw.githubusercontent.com/mrfriman666/mrfriman666/main/SUBA_RUN_V7.ipynb

------------------------------------------------------------------------
Выполняю donor V7: # @title 🏗️ Ячейка 1/5: Создание проекта NLP_Suba_Edition_V7 и конфигов
------------------------------------------------------------------------
Creating project nlp_suba_edition_v7...
Resolving dependencies in `nlp_suba_edition_v7`...
Got dependencies in `nlp_suba_edition_v7`.
Wrote 131 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd nlp_suba_edition_v7
  $ flutter run

Your application code is in nlp_suba_edition_v7/lib/main.dart.

✅ Шаг 1/5 готов: Структура проекта V7 создана!

--------------------------

7135


------------------------------------------------------------------------
Выполняю donor V7: # @title 🔧 SSM2 over CAN + Автоскан PID + Гибрид OBD2
------------------------------------------------------------------------
✅ 1. SSM2 CAN правильный (ISO-TP + автоскан + OBD2 fallback)
✅ 2. OBDService.initECU: автоскан после SSM2 CAN
✅ 3. Terminal: кнопки CAF ON/OFF
✅ 4. Alerts: фильтр мусорных значений

------------------------------------------------------------------------
Выполняю donor V7: # @title 🔧 ФИКС: Умный SSM2 K-Line / CAN автопоиск и ручной выбор
------------------------------------------------------------------------
✅ 1. SubaruSsm2Protocol: K-Line и CAN (без мусора)
✅ 2. Настройки обновлены
✅ 3. Терминал обновлён

------------------------------------------------------------------------
Выполняю donor V7: # @title 🔧 ФИНАЛ: SSM2 CAN + полные настройки + ROM Diff/Запись
------------------------------------------------------------------------
✅ 1. SubaruSsm2Protocol (CAN, вчерашня

<>:1141: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_5838/3886485170.py:1141: SyntaxWarning: invalid escape sequence '\$'
  _snack('CORE PID: \${widget.obdService.activePids.length}', Colors.green);


In [20]:
# @title Ячейка 4/5: Аудит интеграции, зависимости и проверка Dart
import os
import re
import json
import shutil
import subprocess
from pathlib import Path

print('=' * 72)
print('NLP_SE_RUN_V8 | ЯЧЕЙКА 4/5 | АУДИТ И ПОДГОТОВКА')
print('=' * 72)

TARGET = Path('/content/NLP_SE_RUN_V8')
V6_BACKUP = Path('/content/NLP_SE_RUN_V8_V6_BASE')

if not (TARGET / 'lib/main.dart').exists():
    raise FileNotFoundError('Сначала запусти ячейки 1, 2 и 3')

os.chdir(TARGET)

# --- Точечные патчи V8 (идемпотентны; повтор из ячейки 3 для надёжности) ---
shutil.rmtree(TARGET / 'test', ignore_errors=True)
_nk = TARGET / 'lib/protocol/nissan_kwp.dart'
if _nk.exists():
    _s = _nk.read_text(encoding='utf-8', errors='ignore')
    if 'AppConstants' in _s and 'constants.dart' not in _s:
        _lines = _s.splitlines()
        _imports = [i for i, l in enumerate(_lines) if l.lstrip().startswith('import ')]
        _lines.insert((max(_imports) + 1) if _imports else 0,
                      "import '../constants.dart';")
        _nk.write_text('\n'.join(_lines) + '\n', encoding='utf-8')
        print("Патч: import '../constants.dart' добавлен в lib/protocol/nissan_kwp.dart")

# Глобальный патч V8: Object -> dynamic по всему lib/.
# Тип Object у ошибочных PID-значений может быть объявлен в СЕРВИСЕ
# (напр. activePids), а не в дашборде — поэтому чиним глобально.
# В логе ОБЯЗАТЕЛЬНО появится строка «всего замен: N» — это маркер,
# что запущена обновлённая версия ячейки.
_count = 0
for _p in (TARGET / 'lib').rglob('*.dart'):
    _s = _p.read_text(encoding='utf-8', errors='ignore')
    _n = len(re.findall(r'\bObject\b', _s))
    if _n:
        _p.write_text(re.sub(r'\bObject\b', 'dynamic', _s), encoding='utf-8')
        _count += _n
        print('Патч V8: Object -> dynamic:', _p.relative_to(TARGET), '(' + str(_n) + ')')
print('Патч V8: Object -> dynamic, всего замен:', _count)

# Хирургический фикс: тернарник двух PID-библиотек выводит Object (LUB).
# Объявляем элемент цикла dynamic — pid.id/pid.unit компилируются.
_dash = TARGET / 'lib/screens/dashboard_screen.dart'
if _dash.exists():
    _ds = _dash.read_text(encoding='utf-8', errors='ignore')
    _before = _ds
    _ds = _ds.replace('for (final pid in library)',
                      'for (final dynamic pid in library)')
    _ds = re.sub(
        r'final\s+library\s*=\s*profile\.protocol\s*==\s*ProtocolType\.nissanKwp\s*\?\s*NissanPidLibrary\.all\s*:\s*SubaruPidLibrary\.all;',
        'final List<dynamic> library = profile.protocol == ProtocolType.nissanKwp '
        '? NissanPidLibrary.all : SubaruPidLibrary.all;',
        _ds)
    if _ds != _before:
        _dash.write_text(_ds, encoding='utf-8')
        print('Патч V8: dashboard_screen.dart — элемент library объявлен dynamic')
    else:
        print('Патч V8: паттерн library не найден (уже исправлен?)')

# Повторно устанавливаем переменные: Colab иногда теряет PATH после reconnect.
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['PATH'] = (
    '/usr/lib/jvm/java-17-openjdk-amd64/bin:'
    '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:'
    '/content/android-sdk/cmdline-tools/latest/bin:'
    '/content/android-sdk/platform-tools:' + os.environ.get('PATH', '')
)

# Гарантируем имя, версию и SDK после слияния.
pubspec = TARGET / 'pubspec.yaml'
s = pubspec.read_text(encoding='utf-8')
s = re.sub(r'^name:\s*.*$', 'name: nlp_se_run_v8', s, flags=re.M)
s = re.sub(r'^description:\s*.*$',
           'description: Nissan KWP and Subaru SSM2 dual tuning platform',
           s, flags=re.M)
s = re.sub(r'^version:\s*.*$', 'version: 8.0.0+8', s, flags=re.M)
pubspec.write_text(s, encoding='utf-8')

gradle = TARGET / 'android/app/build.gradle.kts'
if gradle.exists():
    g = gradle.read_text(encoding='utf-8')
    g = re.sub(r'namespace\s*=\s*"[^"]+"',
               'namespace = "com.nlp.nlp_se_run_v8"', g)
    g = re.sub(r'applicationId\s*=\s*"[^"]+"',
               'applicationId = "com.nlp.nlp_se_run_v8"', g)
    g = re.sub(r'compileSdk\s*=\s*\d+', 'compileSdk = 36', g)
    g = re.sub(r'targetSdk\s*=\s*\d+', 'targetSdk = 34', g)
    g = re.sub(r'versionCode\s*=\s*\d+', 'versionCode = 8', g)
    g = re.sub(r'versionName\s*=\s*"[^"]+"', 'versionName = "8.0.0"', g)
    g = re.sub(r'\n\s*ndkVersion\s*=.*', '', g)
    g = re.sub(r'(compileSdk\s*=\s*36)',
               '\\1\n    ndkVersion = "28.2.13676358"', g, count=1)
    gradle.write_text(g, encoding='utf-8')

# Инструментальный стек под Flutter 3.47: Gradle >= 8.14, AGP 8.13, Kotlin 2.1.21.
_sg = TARGET / 'android/settings.gradle.kts'
if _sg.exists():
    s = _sg.read_text(encoding='utf-8')
    s = re.sub(r'(id\("com\.android\.application"\)\s*version\s*")[^"]+(")',
               r'\g<1>8.13.0\g<2>', s)
    s = re.sub(r'(id\("org\.jetbrains\.kotlin\.android"\)\s*version\s*")[^"]+(")',
               r'\g<1>2.1.21\g<2>', s)
    _sg.write_text(s, encoding='utf-8')

_wp = TARGET / 'android/gradle/wrapper/gradle-wrapper.properties'
if _wp.exists():
    s = _wp.read_text(encoding='utf-8')
    s = re.sub(r'distributionUrl=.*',
               'distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip',
               s)
    _wp.write_text(s, encoding='utf-8')

_gp = TARGET / 'android/gradle.properties'
if _gp.exists():
    s = _gp.read_text(encoding='utf-8')
    if 'suppressUnsupportedCompileSdk' not in s:
        s += '\nandroid.suppressUnsupportedCompileSdk=36\n'
    _gp.write_text(s, encoding='utf-8')
print('Android-стек: Gradle 8.14 · AGP 8.13.0 · Kotlin 2.1.21 · compileSdk 36 · NDK 28')

# V6-only файлы не должны пропасть. При необходимости возвращаем их.
restored = []
if (V6_BACKUP / 'lib').exists():
    for src in (V6_BACKUP / 'lib').rglob('*.dart'):
        rel = src.relative_to(V6_BACKUP / 'lib')
        dst = TARGET / 'lib' / rel
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            restored.append(str(rel))

print('\n[1/4] flutter pub get')
pub = subprocess.run(
    ['/content/flutter/bin/flutter', 'pub', 'get'],
    cwd=TARGET, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('\n'.join(pub.stdout.splitlines()[-12:]))
if pub.returncode != 0:
    raise RuntimeError('flutter pub get завершился с ошибкой')

print('\n[2/4] dart format')
fmt = subprocess.run(
    ['/content/flutter/bin/dart', 'format', 'lib'],
    cwd=TARGET, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('\n'.join(fmt.stdout.splitlines()[-8:]))

# Диагностический срез: если analyzer снова укажет на dashboard_screen,
# в логе будет виден точный код — для хирургического патча.
_d = TARGET / 'lib/screens/dashboard_screen.dart'
if _d.exists():
    _ls = _d.read_text(encoding='utf-8', errors='ignore').splitlines()
    if len(_ls) > 180:
        print('\nКонтекст dashboard_screen.dart:175-200:')
        for i in range(174, min(200, len(_ls))):
            print('%4d: %s' % (i + 1, _ls[i]))

print('\n[3/4] flutter analyze')
ana = subprocess.run(
    ['/content/flutter/bin/flutter', 'analyze',
     '--no-fatal-infos', '--no-fatal-warnings'],
    cwd=TARGET, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

lines = ana.stdout.splitlines()
error_lines = [line for line in lines if re.search(r'\berror\b', line, flags=re.I)]
warning_lines = [line for line in lines if re.search(r'\bwarning\b', line, flags=re.I)]

print('Ошибок analyzer      :', len(error_lines))
print('Предупреждений       :', len(warning_lines))
if error_lines:
    print('\nОшибки (до 80 строк):')
    for line in error_lines[:80]:
        print(line)
else:
    print('Dart analyzer: ошибок нет')

# Структурный аудит функций. Это не подменяет проверку на реальном ЭБУ,
# но обнаруживает потерю модулей/слоёв во время merge.
print('\n[4/4] структурный аудит V6 + V7')
all_files = list((TARGET / 'lib').rglob('*.dart'))
all_text = '\n'.join(p.read_text(encoding='utf-8', errors='ignore') for p in all_files)
names = {p.name for p in all_files}

checks = {
    # V7
    'ProtocolType': 'ProtocolType' in all_text,
    'Subaru SSM2': ('subaruSsm2' in all_text or 'subaruSsm' in all_text
                    or 'subaru_pid_library.dart' in names),
    'Nissan KWP': 'nissanKwp' in all_text or 'Nissan' in all_text,
    'IAM/FBKC/FKL': all(x in all_text.lower() for x in ['iam', 'fbkc', 'fkl']),
    'выбор protocol в профиле': 'protocol' in all_text.lower() and 'VehicleProfile' in all_text,
    # V6
    'логирование CSV': 'logger_service.dart' in names,
    'анализатор': 'analyzer_service.dart' in names and 'analyzer_screen.dart' in names,
    'DTC': 'dtc_service.dart' in names or 'dtc_screen.dart' in names,
    'терминал': 'terminal_screen.dart' in names,
    'ROM compare': 'rom_compare_screen.dart' in names,
    'performance': 'performance_service.dart' in names or 'performance_screen.dart' in names,
    'карты ECU/ROM': ('rom_map_reader.dart' in names or 'ecu_map_reader.dart' in names),
}

for name, ok in checks.items():
    print(' ', '[OK]' if ok else '[НЕТ]', name)

report = TARGET / 'V8_AUDIT_REPORT.txt'
with report.open('w', encoding='utf-8') as fh:
    fh.write('NLP_SE_RUN_V8 audit\n')
    fh.write('Dart files: ' + str(len(all_files)) + '\n')
    fh.write('Analyzer errors: ' + str(len(error_lines)) + '\n')
    fh.write('Analyzer warnings: ' + str(len(warning_lines)) + '\n\n')
    for name, ok in checks.items():
        fh.write(('[OK] ' if ok else '[MISSING] ') + name + '\n')
    if error_lines:
        fh.write('\nAnalyzer errors:\n' + '\n'.join(error_lines))

missing = [name for name, ok in checks.items() if not ok]

print('\n' + '=' * 72)
print('АУДИТ ЗАВЕРШЁН')
print('Dart-файлов          :', len(all_files))
print('V6-файлов возвращено :', len(restored))
print('Отчёт                :', report)

if missing:
    print('ВНИМАНИЕ: не найдены структурные маркеры:', ', '.join(missing))
if error_lines:
    print('ВНИМАНИЕ: analyzer нашёл ошибки. Ячейка 5 остановит release-сборку,')
    print('пока они не исправлены. Пришли строки ошибок для точечного патча.')
else:
    print('КОД ГОТОВ К RELEASE-СБОРКЕ ЯЧЕЙКОЙ 5')
print('=' * 72)

NLP_SE_RUN_V8 | ЯЧЕЙКА 4/5 | АУДИТ И ПОДГОТОВКА
Патч V8: Object -> dynamic, всего замен: 0
Патч V8: паттерн library не найден (уже исправлен?)
Android-стек: Gradle 8.14 · AGP 8.13.0 · Kotlin 2.1.21 · compileSdk 36 · NDK 28

[1/4] flutter pub get
+ uuid 4.5.0 (4.6.0 available)
  vector_math 2.4.0 (2.4.2 available)
+ vibration 2.0.0 (3.2.1 available)
+ vibration_platform_interface 0.0.3 (0.1.2 available)
+ web 1.1.1
+ win32 5.15.0 (6.4.0 available)
+ win32_registry 2.1.0 (3.0.3 available)
+ xdg_directories 1.1.0
+ yaml 3.1.4
Changed 68 dependencies!
29 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

[2/4] dart format
Formatted lib/services/profile_service.dart
Formatted lib/services/rom_map_reader.dart
Formatted lib/services/rom_saver.dart
Formatted lib/services/settings_service.dart
Formatted lib/services/tuning_service.dart
Formatted lib/widgets/fps_indicator.dart
Formatted lib/widgets/map_table_view.dart
Formatte

In [23]:
# @title Ячейка 5/5: Патч Bluetooth + release APK NLP_SE_RUN_V8
import os
import re
import glob
import shutil
import subprocess
from pathlib import Path

print('=' * 72)
print('NLP_SE_RUN_V8 | ЯЧЕЙКА 5/5 | RELEASE APK')
print('=' * 72)

TARGET = Path('/content/NLP_SE_RUN_V8')
APK_SRC = TARGET / 'build/app/outputs/flutter-apk/app-release.apk'
APK_OUT = Path('/content/NLP_SE_RUN_V8.apk')

if not (TARGET / 'lib/main.dart').exists():
    raise FileNotFoundError('Нет проекта. Запусти ячейки 1-4 по порядку.')

os.chdir(TARGET)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['PATH'] = (
    '/usr/lib/jvm/java-17-openjdk-amd64/bin:'
    '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:'
    '/content/android-sdk/cmdline-tools/latest/bin:'
    '/content/android-sdk/platform-tools:' + os.environ.get('PATH', '')
)

# --- Точечные патчи V8 (идемпотентны; повтор из ячеек 3 и 4) ---
shutil.rmtree(TARGET / 'test', ignore_errors=True)
_nk = TARGET / 'lib/protocol/nissan_kwp.dart'
if _nk.exists():
    _s = _nk.read_text(encoding='utf-8', errors='ignore')
    if 'AppConstants' in _s and 'constants.dart' not in _s:
        _lines = _s.splitlines()
        _imports = [i for i, l in enumerate(_lines) if l.lstrip().startswith('import ')]
        _lines.insert((max(_imports) + 1) if _imports else 0,
                      "import '../constants.dart';")
        _nk.write_text('\n'.join(_lines) + '\n', encoding='utf-8')
        print("Патч: import '../constants.dart' добавлен в lib/protocol/nissan_kwp.dart")

# Глобальный патч V8: Object -> dynamic по всему lib/ (как в ячейках 3 и 4).
# В логе обязана быть строка «всего замен: N» — маркер обновлённой ячейки.
_count = 0
for _p in (TARGET / 'lib').rglob('*.dart'):
    _s = _p.read_text(encoding='utf-8', errors='ignore')
    _n = len(re.findall(r'\bObject\b', _s))
    if _n:
        _p.write_text(re.sub(r'\bObject\b', 'dynamic', _s), encoding='utf-8')
        _count += _n
        print('Патч V8: Object -> dynamic:', _p.relative_to(TARGET), '(' + str(_n) + ')')
print('Патч V8: Object -> dynamic, всего замен:', _count)

# Хирургический фикс дашборда (LUB-тернарник) — см. ячейки 3 и 4.
_dash = TARGET / 'lib/screens/dashboard_screen.dart'
if _dash.exists():
    _ds = _dash.read_text(encoding='utf-8', errors='ignore')
    _before = _ds
    _ds = _ds.replace('for (final pid in library)',
                      'for (final dynamic pid in library)')
    _ds = re.sub(
        r'final\s+library\s*=\s*profile\.protocol\s*==\s*ProtocolType\.nissanKwp\s*\?\s*NissanPidLibrary\.all\s*:\s*SubaruPidLibrary\.all;',
        'final List<dynamic> library = profile.protocol == ProtocolType.nissanKwp '
        '? NissanPidLibrary.all : SubaruPidLibrary.all;',
        _ds)
    if _ds != _before:
        _dash.write_text(_ds, encoding='utf-8')
        print('Патч V8: dashboard_screen.dart — элемент library объявлен dynamic')
    else:
        print('Патч V8: паттерн library не найден (уже исправлен?)')

# Инструментальный стек под Flutter 3.47 (идемпотентно):
# Gradle >= 8.14, AGP 8.13.0, Kotlin 2.1.21, compileSdk 36 + NDK 28.
_g = TARGET / 'android/app/build.gradle.kts'
if _g.exists():
    s = _g.read_text(encoding='utf-8')
    s = re.sub(r'compileSdk\s*=\s*\d+', 'compileSdk = 36', s)
    s = re.sub(r'targetSdk\s*=\s*\d+', 'targetSdk = 34', s)
    s = re.sub(r'namespace\s*=\s*"[^"]+"', 'namespace = "com.nlp.nlp_se_run_v8"', s)
    s = re.sub(r'applicationId\s*=\s*"[^"]+"', 'applicationId = "com.nlp.nlp_se_run_v8"', s)
    s = re.sub(r'versionCode\s*=\s*\d+', 'versionCode = 8', s)
    s = re.sub(r'versionName\s*=\s*"[^"]+"', 'versionName = "8.0.0"', s)
    s = re.sub(r'\n\s*ndkVersion\s*=.*', '', s)
    s = re.sub(r'(compileSdk\s*=\s*36)',
               '\\1\n    ndkVersion = "28.2.13676358"', s, count=1)
    _g.write_text(s, encoding='utf-8')

_sg = TARGET / 'android/settings.gradle.kts'
if _sg.exists():
    s = _sg.read_text(encoding='utf-8')
    s = re.sub(r'(id\("com\.android\.application"\)\s*version\s*")[^"]+(")',
               r'\g<1>8.13.0\g<2>', s)
    s = re.sub(r'(id\("org\.jetbrains\.kotlin\.android"\)\s*version\s*")[^"]+(")',
               r'\g<1>2.2.20\g<2>', s)
    _sg.write_text(s, encoding='utf-8')

_wp = TARGET / 'android/gradle/wrapper/gradle-wrapper.properties'
if _wp.exists():
    s = _wp.read_text(encoding='utf-8')
    s = re.sub(r'distributionUrl=.*',
               'distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip',
               s)
    _wp.write_text(s, encoding='utf-8')

_gp = TARGET / 'android/gradle.properties'
if _gp.exists():
    s = _gp.read_text(encoding='utf-8')
    if 'suppressUnsupportedCompileSdk' not in s:
        s += '\nandroid.suppressUnsupportedCompileSdk=36\n'
    _gp.write_text(s, encoding='utf-8')
print('Патч V8: Gradle 8.14 · AGP 8.13.0 · Kotlin 2.2.20 · compileSdk 36 · NDK 28')

print('\n[1/5] clean + pub get')
subprocess.run(['/content/flutter/bin/flutter', 'clean'], cwd=TARGET,
               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
shutil.rmtree(TARGET / 'build', ignore_errors=True)
pub = subprocess.run(
    ['/content/flutter/bin/flutter', 'pub', 'get'], cwd=TARGET,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('\n'.join(pub.stdout.splitlines()[-10:]))
if pub.returncode != 0:
    raise RuntimeError('flutter pub get failed')

print('\n[2/5] патч устаревших плагинов (bluetooth_serial + vibration)')

# У vibration 2.0.0 AAR собран против android-33, а androidx (core 1.13.1,
# lifecycle 2.7.0) требуют >= 34 — AGP 8.13 падает на checkAarMetadata.
# Лечение — перезаписать build.gradle плагину, как делаем для bluetooth_serial.
PLUGIN_GRADLE_TMPL = """group '%GROUP%'
version '1.0-SNAPSHOT'

buildscript {
    repositories { google(); mavenCentral() }
    dependencies { classpath 'com.android.tools.build:gradle:8.13.0' }
}
allprojects { repositories { google(); mavenCentral() } }
apply plugin: 'com.android.library'

android {
    namespace '%NS%'
    compileSdk 36
    compileOptions {
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }
    defaultConfig { minSdk 21 }
    lintOptions {
        disable 'InvalidPackage'
        checkReleaseBuilds false
        abortOnError false
    }
}
dependencies { implementation 'androidx.core:core:1.13.1' }
"""

BT_MANIFEST = """<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
</manifest>
"""

def pub_cache_glob(pattern):
    hits = (glob.glob('/content/.pub-cache/hosted/pub.dev/' + pattern) +
            glob.glob('/root/.pub-cache/hosted/pub.dev/' + pattern))
    return sorted(set(hits))

def patch_plugin(pattern, group, ns, manifest=None):
    found = pub_cache_glob(pattern)
    if not found:
        print('ВНИМАНИЕ: плагин не найден в кеше:', pattern)
        return
    for folder in found:
        gradle = PLUGIN_GRADLE_TMPL.replace('%GROUP%', group).replace('%NS%', ns)
        Path(folder, 'android/build.gradle').write_text(gradle, encoding='utf-8')
        if manifest is not None:
            mf = Path(folder, 'android/src/main/AndroidManifest.xml')
            mf.parent.mkdir(parents=True, exist_ok=True)
            mf.write_text(manifest, encoding='utf-8')
        print('Пропатчен:', Path(folder).name)

patch_plugin('flutter_bluetooth_serial-*',
             'io.github.edufolly.flutterbluetoothserial',
             'io.github.edufolly.flutterbluetoothserial',
             BT_MANIFEST)
# Глоб с цифрой — чтобы не зацепить vibration_platform_interface
patch_plugin('vibration-[0-9]*',
             'com.benjaminabel.vibration',
             'com.benjaminabel.vibration')

print('\n[3/5] контроль analyzer перед release')
ana = subprocess.run(
    ['/content/flutter/bin/flutter', 'analyze',
     '--no-fatal-infos', '--no-fatal-warnings'],
    cwd=TARGET, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
analysis_lines = ana.stdout.splitlines()
errors = [line for line in analysis_lines if re.search(r'\berror\b', line, flags=re.I)]
print('Analyzer errors:', len(errors))
if errors:
    print('\n'.join(errors[:100]))
    raise RuntimeError(
        'Release остановлен: Dart analyzer нашёл ошибки. '
        'Пришли напечатанные строки ошибок для точечного исправления.'
    )

print('\n[4/5] flutter build apk --release (обычно 8-15 минут)')
build = subprocess.run(
    ['/content/flutter/bin/flutter', 'build', 'apk', '--release'],
    cwd=TARGET, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

for line in build.stdout.splitlines():
    low = line.lower()
    if any(key in low for key in ['error:', 'failed', 'exception', 'built ']) or '✓' in line:
        print(line)

if build.returncode != 0 or not APK_SRC.exists():
    print('\nПОСЛЕДНИЕ 120 СТРОК BUILD LOG:')
    print('=' * 72)
    print('\n'.join(build.stdout.splitlines()[-120:]))
    raise RuntimeError('APK не собран. Пришли последние строки build log.')

print('\n[5/5] копирование и скачивание')
shutil.copy2(APK_SRC, APK_OUT)
size_mb = APK_OUT.stat().st_size / (1024 * 1024)
dart_files = list((TARGET / 'lib').rglob('*.dart'))
dart_lines = sum(len(p.read_text(encoding='utf-8', errors='ignore').splitlines())
                 for p in dart_files)

print('\n' + '=' * 72)
print('NLP_SE_RUN_V8 APK СОБРАН УСПЕШНО')
print('Проект     :', TARGET)
print('APK        :', APK_OUT)
print('Размер     :', f'{size_mb:.1f} MB')
print('Dart-файлы:', len(dart_files))
print('Строк Dart :', dart_lines)
print('=' * 72)

from google.colab import files
files.download(str(APK_OUT))

NLP_SE_RUN_V8 | ЯЧЕЙКА 5/5 | RELEASE APK
Патч V8: Object -> dynamic, всего замен: 0
Патч V8: паттерн library не найден (уже исправлен?)
Патч V8: Gradle 8.14 · AGP 8.13.0 · Kotlin 2.2.20 · compileSdk 36 · NDK 28

[1/5] clean + pub get
  test_api 0.7.12 (0.7.14 available)
  uuid 4.5.0 (4.6.0 available)
  vector_math 2.4.0 (2.4.2 available)
  vibration 2.0.0 (3.2.1 available)
  vibration_platform_interface 0.0.3 (0.1.2 available)
  win32 5.15.0 (6.4.0 available)
  win32_registry 2.1.0 (3.0.3 available)
Got dependencies!
29 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

[2/5] патч устаревших плагинов (bluetooth_serial + vibration)
Пропатчен: flutter_bluetooth_serial-0.4.0
Пропатчен: vibration-2.0.0

[3/5] контроль analyzer перед release
Analyzer errors: 0

[4/5] flutter build apk --release (обычно 8-15 минут)
✓ Built build/app/outputs/flutter-apk/app-release.apk (58.8MB)

[5/5] копирование и скачивание

NLP_SE_RUN_V8

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>